# 2. Oxygen adsorption on Cu(111)

**Kernel:** MACE.
Run cells from top to bottom in a fresh kernel. Each setup creates a new results directory.

**Learning goals:** build a slab; constrain substrate layers; compare adsorption geometries and energies

**Working pattern:** predict a result, run the calculation, inspect the geometry and convergence,
Energy is reported in eV, length in angstrom, and force in eV/angstrom.


In [1]:
from pathlib import Path
import sys
# Works when Jupyter starts in the repository, workshop folder, or exercise folder.
_candidates = [Path.cwd(), *Path.cwd().parents]
WORKSHOP = next((p for base in _candidates for p in (base, base / 'workshop_demo')
                 if (p / 'workshop_utils.py').is_file()), None)
if WORKSHOP is None:
    raise RuntimeError('Launch Jupyter from the repository or workshop_demo folder.')
if str(WORKSHOP) not in sys.path:
    sys.path.insert(0, str(WORKSHOP))
from workshop_utils import start_exercise, mace_model, relax, smoke_check, signed_angle
DATA, OUTPUT = start_exercise('InorganicCrystals')
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from ase.visualize import view


Python: /home/nchopper/anaconda3/envs/mace_demo/bin/python
Inputs: /home/nchopper/mlip-demo/workshop_demo/InorganicCrystals
New results: /home/nchopper/mlip-demo/workshop_demo/InorganicCrystals/results/20260923-103358-cb5e06


## 1. Build the substrate
We use a small slab for workshop speed, not a converged surface model. The bottom layer is fixed;
the other atoms can relax. All site calculations use the same cell, vacuum, and constraints.
Predict the preferred site: ontop, bridge, fcc, or hcp. A starting site may change during relaxation.

For this exercise, we will be using the Atomic Simulation Environment's (ASE) built-in builder tools. fcc111 returns an ase.Atoms object for a face-centered cubic cell with the (111) miller indice surface


In [ ]:
from ase.build import fcc111, add_adsorbate
from ase.constraints import FixAtoms
from ase.optimize import BFGS

# Create a Cu(111) slab with 3 layers and a vacuum of 10 Å
slab = fcc111('Cu', size=(2, 2, 3), vacuum=10.0)

# Fix the bottom layer of the slab to simulate a surface with a bulk-like region
#   slab.get_tags() returns an array of integers representing the layer index of each atom in the slab.
#   The bottom layer is identified as the layer with the maximum tag value.
#      So slab.get_tags() == slab.get_tags().max() creates a boolean array 
#      where True corresponds to atoms in the bottom layer.
slab.set_constraint(FixAtoms(mask=slab.get_tags() == slab.get_tags().max()))

view(slab, viewer='x3d')

[3 3 3 3 2 2 2 2 1 1 1 1]


In [9]:
from mace.calculators import MACECalculator
DEVICE = 'cpu'  # Only select cuda inside a GPU allocation with a compatible environment.
DEFAULT_DTYPE = 'float32'  # Faster but less accurate than float64
calculator = MACECalculator(model_paths=mace_model('2023-12-03-mace-128-L1_epoch-199.model'),
                            device=DEVICE, default_dtype=DEFAULT_DTYPE)


/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


MACE checkpoint: /home/nchopper/mlip-demo/workshop_demo/models/2023-12-03-mace-128-L1_epoch-199.model


In [10]:
slab.calc = calculator
smoke_check(slab)
slab_ok = relax(slab, BFGS, fmax=0.05, steps=150, logfile=str(OUTPUT / 'clean_slab.log'))
write(OUTPUT / 'clean_slab.extxyz', slab)


Energy: -45.351448 eV; maximum force: 0.0349 eV/angstrom
Converged: True; steps: 0; target: 0.05 eV/angstrom


/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/ase/io/extxyz.py:320: UserWarning: Skipping unhashable information adsorbate_info
  warnings.warn('Skipping unhashable information '


## 2. Relax each starting site
For a short exercise use `['fcc', 'hcp']`; include all four sites for the extension.
A 0.05 eV/angstrom threshold is a workshop setting. Tighten it later to test numerical sensitivity.
Inspect the final geometry rather than assuming its name still identifies its site.


In [ ]:
sites = ['ontop', 'bridge', 'fcc', 'hcp']
systems, records = {}, []
for site in sites:
    system = slab.copy()
    add_adsorbate(system, 'O', height=1.5, position=site)
    # Reset the constraint after adding the adsorbate.
    system.set_constraint(FixAtoms(indices=np.flatnonzero(slab.get_tags() == slab.get_tags().max())))
    system.calc = calculator
    ok = relax(system, BFGS, fmax=0.05, steps=150, logfile=str(OUTPUT / f'{site}.log'))
    systems[site] = system
    records.append((site, system.get_potential_energy(), ok))
    write(OUTPUT / f'{site}.extxyz', system)
reference_energy = min(energy for _, energy, ok in records if ok) if any(r[2] for r in records) else None
for site, energy, ok in records:
    relative = energy - reference_energy if ok else float('nan')
    print(f'{site:8s} converged={ok}, relative energy={relative:.4f} eV')


In [ ]:
site_to_inspect = 'fcc'
view(systems[site_to_inspect], viewer='x3d')


## 3. What do these energies mean?
The primary comparison is between relaxed cells with **identical composition**. Their energy
differences do not require an isolated-oxygen reference. Exclude unconverged structures from ranking.
If two starting sites relax to the same geometry, they are not two distinct adsorption minima.

An absolute adsorption energy additionally needs a physically appropriate reference:

$$E_{ads}=E_{slab+O}-E_{slab}-E_{reference}.$$

Atomic O and half an O2 molecule define different quantities. Validate the model for the chosen
reference, including electronic state, before comparing to experiment. The older MACE-MP-0
checkpoint used here should not be assumed accurate for isolated atoms merely because it runs.
See the [MACE model notes](https://github.com/ACEsuit/mace-foundations).

## Try, explain, and report
- Which starting sites remain distinct? Did bridge move into a hollow site?
- Repeat with a tighter force threshold. Does the ranking change?
- For an advanced convergence study, increase slab thickness and lateral size separately.
  Increasing lateral size with one oxygen also changes coverage; explain that distinction.

**Checkpoint:** report relative site energies, convergence flags, final geometries, and the fixed layers.
Do not present the small slab calculation as an experimentally validated binding energy.
